In [1]:
import os, pickle, warnings
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
import ee
from coastsat import SDS_download, SDS_preprocess, SDS_shoreline, SDS_tools
ee.Initialize(project='tefesat')
print("OK")

OK


In [2]:
# Zone : nord de la Langue de Barbarie (Guet Ndar, Goxu Mbacc, Ndar Toute)
# Format : [longitude, latitude]
polygon = [[[-16.54, 15.97], [-16.47, 15.97], [-16.47, 16.06],
            [-16.54, 16.06], [-16.54, 15.97]]]
polygon = SDS_tools.smallest_rectangle(polygon)

inputs = {
    'polygon': polygon,
    'dates': ['2015-07-01', '2025-12-31'],
    'sat_list': ['S2'],
    'sitename': 'SAINT_LOUIS_NORD',
    'filepath': os.path.join(os.getcwd(), 'data'),
    'landsat_collection': 'C02'
}
print("Zone et dates définies")

Zone et dates définies


In [3]:
result = SDS_download.check_images_available(inputs)
metadata = SDS_download.retrieve_images(inputs)
metadata = SDS_download.get_metadata(inputs)

Number of images available between 2015-07-01 and 2025-12-31:
- In Landsat Tier 1 & Sentinel-2 Level-1C:
     S2: 680 images
  Total to download: 680 images
S2: 680/680 images already exist, 0 to be downloaded
Number of images available between 2015-07-01 and 2025-12-31:
- In Landsat Tier 1 & Sentinel-2 Level-1C:
     S2: 680 images
  Total to download: 680 images
S2: 680/680 images already exist, 0 to be downloaded

S2: 0 images

Satellite images downloaded from GEE and save in C:\MonCarriere\tefeSAT\Hackaton_Sensat\CoastSat\data\SAINT_LOUIS_NORD


In [4]:
settings = {
    'cloud_thresh': 0.1, 'dist_clouds': 300, 'output_epsg': 32628,
    'check_detection': False, 'adjust_detection': False, 'save_figure': True,
    'min_beach_area': 1000, 'min_length_sl': 500, 'cloud_mask_issue': False,
    'sand_color': 'default', 'pan_off': False, 's2cloudless_prob': 40,
    'inputs': inputs
}
#SDS_preprocess.save_jpg(metadata, settings, use_matplotlib=True)

In [6]:
%matplotlib qt
settings['reference_shoreline'] = SDS_preprocess.get_reference_sl(metadata, settings)
settings['max_dist_ref'] = 100

Reference shoreline has been saved in C:\MonCarriere\tefeSAT\Hackaton_Sensat\CoastSat\data\SAINT_LOUIS_NORD


In [8]:
%matplotlib qt
settings['check_detection'] = False
output = SDS_shoreline.extract_shorelines(metadata, settings)
output = SDS_tools.remove_duplicates(output)
output = SDS_tools.remove_inaccurate_georef(output, 10)
print("Traits de côte extraits :", len(output['dates']))

Mapping shorelines:
S2:   100%
17 duplicates
3 bad georef
Traits de côte extraits : 431


In [9]:
with open(os.path.join(inputs['filepath'], inputs['sitename'], 'output_sauvegarde.pkl'), 'wb') as f:
    pickle.dump(output, f)
print("Sauvegarde OK :", len(output['dates']), "traits de côte")

Sauvegarde OK : 431 traits de côte


In [15]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString

rows = []
for i, d in enumerate(output['dates']):
    sl = np.array(output['shorelines'][i])
    if sl.size == 0:
        continue
    sl = sl[~np.isnan(sl).any(axis=1)]
    if len(sl) < 2:
        continue
    ligne = LineString(sl)
    rows.append({'date': d.strftime('%Y-%m-%d'),
                 'satellite': output['satname'][i],
                 'precision_m': output['geoaccuracy'][i],
                 'longueur_m': ligne.length,
                 'geometry': ligne})

gdf = gpd.GeoDataFrame(rows, crs='EPSG:32628')
gdf = gdf.sort_values('longueur_m').drop_duplicates('date', keep='last').sort_values('date')
med = gdf['longueur_m'].median()
gdf['ok'] = gdf['longueur_m'].between(0.6 * med, 1.4 * med)

print("Traits de côte :", len(gdf), "| écartés (longueur anormale) :", (~gdf['ok']).sum())
print("\nNombre de dates par année :")
print(gdf[gdf['ok']].groupby(gdf['date'].str[:4]).size())

Traits de côte : 427 | écartés (longueur anormale) : 56

Nombre de dates par année :
date
2015     2
2016    17
2017    30
2018    36
2019    41
2020    39
2021    32
2022    36
2023    45
2024    43
2025    50
dtype: int64


In [ ]:
%matplotlib inline
bon = gdf[gdf['ok']].copy()
bon['annee'] = bon['date'].str[:4].astype(int)
fig, ax = plt.subplots(figsize=(6, 9))
bon.plot(ax=ax, column='annee', cmap='viridis', linewidth=0.6, legend=True)
ax.set_title("Traits de côte 2015-2025 (couleur = année)")
plt.show()